In [ ]:
import asyncio
import csv
from datetime import datetime
from asyncua import Client
import nest_asyncio
import os
nest_asyncio.apply()

filnavn = "PID"
CSV_FIL = f"{filnavn}.csv"
OPC_URL = "opc.tcp://127.0.0.1:49320"

INTERVALL_SEK = 0.05

TAGS = {
    "pådrag":     "ns=2;s=Rigg1.PLS1.flow_glob",
    "tank":       "ns=2;s=Rigg1.PLS1.TankNivå",
}

async def main():
    try:
        # Opprett klient
        client = Client(url=OPC_URL)
        
        # Bruk bare serveren sitt sertifikat for validering
        # Format: policy,mode,client_cert,client_key,server_cert
        server_cert = "server_cert.pem"
        
        if os.path.exists(server_cert):
            # Sett sikkerhet med bare server-sertifikat (ingen klient-sertifikat nødvendig)
            await client.set_security_string(
                f"Basic256Sha256,SignAndEncrypt,,,{server_cert}"
            )
            print(f"Sikkerhet konfigurert med server-sertifikat")
        else:
            print(f"Advarsel: Server-sertifikatfil ikke funnet")
            print(f"  {server_cert}: Ikke funnet")
        
        async with client:
            print(f"Koblet til {OPC_URL}")

            noder = {navn: client.get_node(adresse) for navn, adresse in TAGS.items()}

            with open(CSV_FIL, mode="w", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow(["Tidsstempel"] + list(TAGS.keys()))

                for i in range(100000):  # Endre til while True for evig kjøring
                    tidsstempel = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    verdier = []

                    for navn, node in noder.items():
                        try:
                            verdi = await node.read_value()
                            verdier.append(verdi)
                        except Exception as e:
                            print(f"Feil ved lesing av '{navn}': {e}")
                            verdier.append(None)

                    writer.writerow([tidsstempel] + verdier)
                    f.flush()
                    print(f"[{tidsstempel}] {dict(zip(TAGS.keys(), verdier))}")
                    if i % 20 == 0:  # Vis status hver 20. lesing
                        print(f"  ... {i} lesinger gjennomført")
                    await asyncio.sleep(INTERVALL_SEK)

        print(f"Ferdig! Data lagret i '{CSV_FIL}'")
        
    except Exception as e:
        print(f"Feil: {e}")
        print(f"Sørg for at OPC-serveren kjører på {OPC_URL}")
        import traceback
        traceback.print_exc()

asyncio.run(main())

Feil: Valid PEM but no BEGIN/END delimiters for a private key found. Are you sure this is a private key?
Sørg for at OPC-serveren kjører på opc.tcp://127.0.0.1:49320


Traceback (most recent call last):
  File "C:\Users\noast\AppData\Local\Temp\ipykernel_10968\99084604.py", line 32, in main
    await client.set_security_string(
  File "C:\Users\noast\AppData\Roaming\Python\Python312\site-packages\asyncua\client\client.py", line 189, in set_security_string
    return await self.set_security(
           ^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\noast\AppData\Roaming\Python\Python312\site-packages\asyncua\client\client.py", line 227, in set_security
    return await self._set_security(policy, certificate, private_key, server_certificate, mode)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\noast\AppData\Roaming\Python\Python312\site-packages\asyncua\client\client.py", line 240, in _set_security
    pk = await uacrypto.load_private_key(
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\noast\AppData\Roaming\Python\Python312\site-packages\asyncua\crypto\uacrypto.py", line 128, in l

In [37]:
# Konvertér DER-sertifikater til PEM-format
from cryptography import x509
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import serialization

# Last DER-sertifikatene (erstatt med faktiske filstier hvis nødvendig)
# Antar at filene ligger i samme mappe
try:
    # Les de binære DER-filene
    with open("6f7de7a25af6551f29e5470b7e6ba56881f41caa.der", "rb") as f:
        server_der = f.read()
    
    with open("2b8f198499f245b3f04d735df429a9ae787a509c.der", "rb") as f:
        client_der = f.read()
    
    # Last og konvertér til PEM
    server_cert = x509.load_der_x509_certificate(server_der, default_backend())
    client_cert = x509.load_der_x509_certificate(client_der, default_backend())
    
    # Skriv som PEM-filer
    with open("server_cert.pem", "wb") as f:
        f.write(server_cert.public_bytes(serialization.Encoding.PEM))
    
    with open("client_cert.pem", "wb") as f:
        f.write(client_cert.public_bytes(serialization.Encoding.PEM))
    
    print("✓ Sertifikater konvertert til PEM-format")
    print("✓ Lagret: server_cert.pem")
    print("✓ Lagret: client_cert.pem")
    
except FileNotFoundError as e:
    print(f"Feil: Kunne ikke finne sertifikatfiler - {e}")
    print("Sørg for at DER-filene ligger i samme mappe som notebooken")

✓ Sertifikater konvertert til PEM-format
✓ Lagret: server_cert.pem
✓ Lagret: client_cert.pem


C:\Users\noast\AppData\Local\Temp\ipykernel_10968\3811784080.py:17: CryptographyDeprecationWarning: Parsed a serial number which wasn't positive (i.e., it was negative or zero), which is disallowed by RFC 5280. Loading this certificate will cause an exception in a future release of cryptography.
  server_cert = x509.load_der_x509_certificate(server_der, default_backend())
